In [1]:
# %pip install spark-nlp
import json
import pandas as pd
import numpy as np

import sparknlp
import pyspark.sql.functions as F

from pyspark.ml import Pipeline
from pyspark.sql import SparkSession
from sparknlp.annotator import *
from sparknlp.base import *
from sparknlp.pretrained import PretrainedPipeline
from pyspark.sql.types import StringType, IntegerType

In [2]:
spark = (
  SparkSession
  .builder
  .appName('SpeechSentiment')
  # .master('local[*]')
  # .config('spark.driver.memory', '16G')
  .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
  .config('spark.kryoserializer.buffer.max', '2000M')
  .config('spark.jars.packages', 'com.johnsnowlabs.nlp:spark-nlp-silicon_2.13:' +sparknlp.version())
  .getOrCreate()
)
spark


:: loading settings :: url = jar:file:/Users/jpj/.pyenv/versions/3.14.5/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/jpj/.ivy2.5.2/cache
The jars for the packages stored in: /Users/jpj/.ivy2.5.2/jars
com.johnsnowlabs.nlp#spark-nlp-silicon_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c59aac3c-f96e-4c9d-b9f4-3b219a7f9cf0;1.0
	confs: [default]
	found com.johnsnowlabs.nlp#spark-nlp-silicon_2.13;7.0.1 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central
	found com.amazonaws#aws-java-sdk-s3;1.12.500 in central
	found com.amazonaws#aws-java-sdk-kms;1.12.500 in central
	found com.amazonaws#aws-java-sdk-core;1.12.500 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apach

In [ ]:
# Enter examples as strings in this array
input_list = [
    """From the beginning, we were met by friendly staff members, and the convienent parking at Chelsea Piers made it easy for us to get to the boat.""",
    """From the beginning, we were met by evil, terrible staff members! The inconvienent parking at Chelsea Piers made it grueling for us to get to the boat."""
]

In [12]:
document_assembler = DocumentAssembler() \
    .setInputCol('text')\
    .setOutputCol('document')

sentence_detector = SentenceDetector() \
    .setInputCols(['document'])\
    .setOutputCol('sentence')

tokenizer = Tokenizer()\
    .setInputCols(['sentence']) \
    .setOutputCol('token')

word_embeddings = WordEmbeddingsModel.pretrained("glove_6B_300", "xx")\
    .setInputCols(["document", "token"])\
    .setOutputCol("embeddings")

ner_model = NerDLModel.pretrained("ner_aspect_based_sentiment")\
    .setInputCols(["document", "token", "embeddings"])\
    .setOutputCol("ner")

ner_converter = NerConverter()\
    .setInputCols(['sentence', 'token', 'ner']) \
    .setOutputCol('ner_chunk')

glove_6B_300 download started this may take some time.
Approximate size to download 426.2 MB
ner_aspect_based_sentiment download started this may take some time.
Approximate size to download 21.3 MB


In [13]:
nlp_pipeline = Pipeline(
    stages=[
      document_assembler,
      sentence_detector,
      tokenizer,
      word_embeddings,
      ner_model,
      ner_converter])

df = spark.createDataFrame(input_list, StringType()).toDF("text")
result = nlp_pipeline.fit(df).transform(df)

In [14]:
# result.show(truncate=40)

In [15]:
# Using display lib
from sparknlp_display import NerVisualizer

NerVisualizer().display(result.collect()[0], 'ner_chunk', 'document')

26/11/26 12:30:49 WARN DAGScheduler: Broadcasting large task binary with size 1255.5 KiB


In [16]:
# Using display lib
from sparknlp_display import NerVisualizer

NerVisualizer().display(result.collect()[1], 'ner_chunk', 'document')

26/11/26 12:31:00 WARN DAGScheduler: Broadcasting large task binary with size 1255.5 KiB
